In [ ]:
import yaml
import argparse
from tqdm import tqdm
from pathlib import Path
from copy import deepcopy
from datetime import datetime
import itertools

import numpy as np
import matplotlib.pyplot as plt
import torch
from torchmetrics import MeanMetric
from torch.utils.tensorboard import SummaryWriter
from torch_geometric.utils import unbatch


In [ ]:


from utils import set_seed, get_optimizer, log, get_lr_scheduler, add_random_edge, get_loss
from utils.get_data import get_data_loader, get_dataset
from utils.get_model import get_model
from utils.metrics import acc_and_pr_at_k, point_filter
import time
import math
import logging

In [ ]:
%ls ~

In [ ]:

def setup_logger(log_dir):
    logger = logging.getLogger('eval_result')
    logger.setLevel(logging.INFO)
    
    # Create handlers
    c_handler = logging.StreamHandler()
    f_handler = logging.FileHandler(log_dir / 'eval_result.txt')
    c_handler.setLevel(logging.INFO)
    f_handler.setLevel(logging.INFO)
    
    # Create formatters and add them to handlers
    c_format = logging.Formatter('%(message)s')
    f_format = logging.Formatter('%(message)s')
    c_handler.setFormatter(c_format)
    f_handler.setFormatter(f_format)
    
    # Add handlers to the logger
    logger.addHandler(c_handler)
    logger.addHandler(f_handler)
    
    return logger




@torch.no_grad()
def eval_one_batch(model, optimizer, criterion, data, lr_s):
    model.eval()
    embeddings = model(data)
    #loss = criterion(embeddings, data.point_pairs_index, data.particle_id, data.reconstructable, data.pt)
    loss = criterion(embeddings, data.edge_index, data.particle_id, data.reconstructable, data.pt)
    return loss.item(), embeddings.detach(), data.particle_id.detach()

def process_data(data, phase, device, epoch, p=0.2):
    data = data.to(device)
    return data

In [ ]:
def run_one_epoch(config, model, optimizer, criterion, data_loader, phase, epoch, device, metrics, lr_s):
    run_one_batch = eval_one_batch
    phase = "test " if phase == "test" else phase
    
    log_dir = Path(config['log_dir'])
    log_dir.mkdir(parents=True, exist_ok=True)  # Ensure the directory exists
    
    logger = setup_logger(log_dir)
    
    #pt_bins = [0, 0.2, 0.5, 0.9, 1.2, 1.5, 2, 3]
    pt_bins = [0, 0.2, 0.5, 0.9, 1.2, 1.5, 2, 3, 4, 5]
    
    pbar = tqdm(data_loader, disable=__name__ != "__main__")
    for idx, data in enumerate(pbar):
        data = process_data(data, phase, device, epoch)
        
        # Capture all returned values including additional metrics
        batch_loss, batch_embeddings, batch_cluster_ids = run_one_batch(model, optimizer, criterion, data, lr_s)

        # Original metrics
        batch_acc2,batch_acc09,batch_rec2,batch_rec09,batch_eff2,batch_eff09, batch_efflhc2,batch_efflhc09 = update_metrics(metrics, data, batch_embeddings, batch_cluster_ids, criterion.dist_metric)
        metrics["loss"].update(batch_loss)

        # New information log (FLOPs, throughput, inference time)
        desc = (f"{phase} acc@2: {batch_acc2:.4f}, acc@0.9: {batch_acc09:.4f}, "
                f"rec@2: {batch_rec2:.4f}, rec@0.9: {batch_rec09:.4f}, "
                f"eff_dm@0.9: {batch_eff09:.4f}, eff_lhc@0.9: {batch_efflhc09:.4f}, "
        pbar.set_description(desc)
        logger.info(desc)
        
    # Check if any updates have been made to the metrics
    metric_res = compute_metrics(metrics) if any(metric._update_called for metric in metrics.values()) else None
    
    if metric_res:
        loss, acc = (metric_res["loss"], metric_res["accuracy@0.9"])
        prec, recall = metric_res["precision@0.9"], metric_res["recall@0.9"]
        desc = f"[Epoch {epoch}] {phase}, loss: {loss:.4f}, acc: {acc:.4f}, prec: {prec:.4f}, recall: {recall:.4f}"
        logger.info(desc)
    else:
        desc = f"[Epoch {epoch}] {phase}, no updates made to metrics."
        logger.info(desc)
    
    pbar.set_description(desc)
    logger.info(desc)
    
    if phase.strip() == "test":
        pass
    
    return metric_res
        

In [ ]:
def reset_metrics(metrics):
    for metric in metrics.values():
        if isinstance(metric, MeanMetric):
            metric.reset()

def compute_metrics(metrics):
    return {
        f"{name}@{pt}": metrics[f"{name}@{pt}"].compute().item()
        for name in ["accuracy", "precision", "recall", "efficiency_dm", "efficiency_lhc"]
        for pt in metrics["pt_thres"]
    } | {"loss": metrics["loss"].compute().item()}

def update_metrics(metrics, data, batch_embeddings, batch_cluster_ids, dist_metric):
    embeddings = unbatch(batch_embeddings, data.batch)
    cluster_ids = unbatch(batch_cluster_ids, data.batch)

    acc_2, acc_09, rec_2, rec_09 = 0, 0, 0, 0
    eff_2, eff_09, eff_lhc_2, eff_lhc_09 = 0, 0, 0, 0  

    for pt in metrics["pt_thres"]:
        batch_mask = point_filter(batch_cluster_ids, data.reconstructable, data.pt, pt_thres=pt)
        mask = unbatch(batch_mask, data.batch)

        if all(len(m) == 0 for m in mask):  # Check if mask is empty for all batches
            continue  # Skip this iteration if mask is empty

        res = []
        for i in range(len(embeddings)):
            if len(mask[i]) == 0:  # Skip if mask[i] is empty
                continue
            try:
                acc_pr_eff_res = acc_and_pr_at_k(embeddings[i], cluster_ids[i], mask[i], dist_metric)
                res.append(acc_pr_eff_res)
            except ValueError as e:
                print(f"Skipping due to error: {e}")
                continue

        if res: 
            res = torch.tensor(res)
            
            if torch.isnan(res).any():
                print(f"Skipping update due to NaN values in results for pt={pt}")
                continue
            metrics[f"accuracy@{pt}"].update(res[:, 0])
            metrics[f"precision@{pt}"].update(res[:, 1])
            metrics[f"recall@{pt}"].update(res[:, 2])
            metrics[f"efficiency_dm@{pt}"].update(res[:, 3])  # Update efficiency
            metrics[f"efficiency_lhc@{pt}"].update(res[:, 4]) 

            if pt == 2:
                acc_2 = res[:, 0].mean().item()
                rec_2 = res[:, 2].mean().item()
                eff_2 = res[:, 3].mean().item()  # Efficiency for pt=2
                eff_lhc_2 = res[:, 4].mean().item()  # Efficiency for pt=2
            if pt == 0.9:
                acc_09 = res[:, 0].mean().item()
                rec_09 = res[:, 2].mean().item()
                eff_09 = res[:, 3].mean().item()  # Efficiency for pt=0.9
                eff_lhc_09 = res[:, 4].mean().item()  # Efficiency for pt=2

    return acc_2,  acc_09, rec_2, rec_09, eff_2,  eff_09, eff_lhc_2, eff_lhc_09


def run_one_seed(config):
    device = torch.device(config["device"] if torch.cuda.is_available() else "cpu")
    torch.set_num_threads(config["num_threads"])

    dataset_name = config["dataset_name"]
    model_name = config["model_name"]
    dataset_dir = Path(config["data_dir"]) / dataset_name.split("-")[0]
    log(f"Device: {device}, Model: {model_name}, Dataset: {dataset_name}, Note: {config['note']}")

    time = datetime.now().strftime("%m_%d-%H_%M_%S.%f")[:-4]
    rand_num = np.random.randint(10, 100)
    log_dir = Path(config['log_dir'])
    log(f"Log dir: {log_dir}")
    log_dir.mkdir(parents=True, exist_ok=True)  # Ensure the directory exists
    
    logger = setup_logger(log_dir)
    
    writer = SummaryWriter(log_dir) if config["log_tensorboard"] else None

    set_seed(config["seed"])
    dataset = get_dataset(dataset_name, dataset_dir)
    loaders = get_data_loader(dataset, dataset.idx_split, batch_size=config["batch_size"])

    model = get_model(model_name, config["model_kwargs"], dataset)

    model_path = config["model_path"]
    checkpoint = torch.load(model_path, map_location="cpu")
    model.load_state_dict(checkpoint, strict=True)
    model = model.to(device)

    opt = get_optimizer(model.parameters(), config["optimizer_name"], config["optimizer_kwargs"])
    config["lr_scheduler_kwargs"]["num_training_steps"] = 1 * len(loaders["train"])
    lr_s = get_lr_scheduler(opt, config["lr_scheduler_name"], config["lr_scheduler_kwargs"])
    criterion = get_loss(config["loss_name"], config["loss_kwargs"])

    main_metric = config["main_metric"]
    pt_thres = [0, 0.9, 1.2, 1.5, 2, 3]
    metric_names = ["accuracy", "precision", "recall", "efficiency_dm", "efficiency_lhc"]
    metrics = {f"{name}@{pt}": MeanMetric(nan_strategy="error") for name in metric_names for pt in pt_thres}
    metrics["loss"] = MeanMetric(nan_strategy="error")
    metrics["pt_thres"] = pt_thres

    coef = 1 if config["mode"] == "max" else -1
    best_epoch = 0
    best_valid, best_test = deepcopy(metrics), deepcopy(metrics)

    if writer is not None:
        layout = {
            "Gap": {
                "loss": ["Multiline", ["valid/loss", "test/loss"]],
                "acc@0.9": ["Multiline", ["valid/accuracy@0.9", "test/accuracy@0.9"]],
            }
        }
        writer.add_custom_scalars(layout)

    # Evaluate on the validation set
    valid_res = run_one_epoch(config, model, opt, criterion, loaders["valid"], "valid", 0, device, metrics, lr_s)

    # Evaluate on the test set
    test_res = run_one_epoch(config, model, opt, criterion, loaders["test"], "test", 0, device, metrics, lr_s)

    if valid_res and (valid_res[main_metric] * coef) > (best_valid[main_metric].compute().item() * coef):
        best_valid, best_test = valid_res, test_res


    
    print("=" * 50)
    print("=" * 50)

In [ ]:
run_one_seed(yaml.safe_load(Path('./configs/tracking/tracking_mamba_hydra.yaml').open("r").read()))